## 1. Imports

In [ ]:
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', None)


In [ ]:
raw_csv = """Name,Age,City,Salary,JoinDate,Gender
Asha,25,Bengaluru,55000,2021-05-12,F
Ravi,30, delhi ,62000,2020-11-03,M
Meena,,Mumbai,58000,2019-07-23,F
Karan,28,BENGALURU,,2022-01-15,M
Asha,25,Bengaluru,55000,2021-05-12,F
Divya,120,Chennai,60000,2020-03-30,F
Suresh,35,Delhi,59000,not_a_date,M
Priya,29,,61000,2021-09-09,F
Anil,40,Mumbai,150000,2018-06-01,M
Zoya,,chennai,57000,2022-02-20,F
"""

with open("sample_raw_data.csv", "w") as f:
    f.write(raw_csv)

df = pd.read_csv("sample_raw_data.csv")
df


## 3. Initial Exploration

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.describe(include='all')

In [ ]:
# Missing values per column
df.isnull().sum()

In [ ]:
# Percentage missing per column
(df.isnull().mean() * 100).round(1)

## 4. Handle Duplicates

Check for and remove exact duplicate rows (e.g. "Asha" appears twice with identical values).

In [ ]:
print("Duplicate rows:", df.duplicated().sum())
df = df.drop_duplicates().reset_index(drop=True)
df

## 5. Standardize Text / Categorical Columns

`City` has inconsistent casing and extra whitespace (`" delhi "`, `"BENGALURU"`). Standardize to a consistent title-case format.

In [ ]:
df["City"] = df["City"].str.strip().str.title()
df["City"].unique()

## 6. Handle Missing Values

- `Age`: numeric → impute with **median** (robust to the outlier age of 120)
- `Salary`: numeric → impute with **mean**
- `City`: categorical → impute with **mode** (most frequent city)
- `JoinDate`: will be fixed after converting to datetime (invalid parses become NaT, then we decide how to handle)

In [ ]:
df["Age"] = df["Age"].fillna(df["Age"].median())
df["Salary"] = df["Salary"].fillna(df["Salary"].mean())
df["City"] = df["City"].fillna(df["City"].mode()[0])

df.isnull().sum()

## 7. Fix Data Types

In [ ]:
df["Age"] = df["Age"].astype(int)
df["Salary"] = df["Salary"].astype(float)

# Convert JoinDate; invalid strings like "not_a_date" become NaT
df["JoinDate"] = pd.to_datetime(df["JoinDate"], errors="coerce")

df.dtypes

In [ ]:
# Rows where date failed to parse
df[df["JoinDate"].isnull()]

In [ ]:
# Fill any resulting NaT with the median join date (a reasonable placeholder strategy)
median_date = df["JoinDate"].median()
df["JoinDate"] = df["JoinDate"].fillna(median_date)
df

## 8. Handle Outliers (IQR method)

`Age` has an implausible value (120) and `Salary` has a very high outlier (150000). Use the IQR rule to detect and cap/remove outliers.

In [ ]:
def iqr_bounds(series):
    Q1 = series.quantile(0.25)
    Q3 = series.quantile(0.75)
    IQR = Q3 - Q1
    return Q1 - 1.5 * IQR, Q3 + 1.5 * IQR

age_low, age_high = iqr_bounds(df["Age"])
sal_low, sal_high = iqr_bounds(df["Salary"])

print(f"Age bounds: {age_low:.1f} to {age_high:.1f}")
print(f"Salary bounds: {sal_low:.1f} to {sal_high:.1f}")

In [ ]:
# Flag outliers rather than silently dropping — inspect before deciding
df["Age_outlier"] = ~df["Age"].between(age_low, age_high)
df["Salary_outlier"] = ~df["Salary"].between(sal_low, sal_high)
df[["Name","Age","Age_outlier","Salary","Salary_outlier"]]

In [ ]:
# Cap outliers to the bound values (winsorizing) instead of dropping rows
df["Age"] = df["Age"].clip(lower=age_low, upper=age_high).astype(int)
df["Salary"] = df["Salary"].clip(lower=sal_low, upper=sal_high)

df = df.drop(columns=["Age_outlier", "Salary_outlier"])
df

## 9. Filtering Examples

In [ ]:
# Employees older than 28
df[df["Age"] > 28]

In [ ]:
# Employees in Bengaluru or Mumbai
df[df["City"].isin(["Bengaluru", "Mumbai"])]

In [ ]:
# Female employees earning above the average salary
df[(df["Gender"] == "F") & (df["Salary"] > df["Salary"].mean())]

## 10. GroupBy Examples

In [ ]:
# Average salary by city
df.groupby("City")["Salary"].mean().round(2)

In [ ]:
# Multiple aggregations by gender
df.groupby("Gender").agg(
    avg_age=("Age", "mean"),
    avg_salary=("Salary", "mean"),
    count=("Name", "count")
).round(1)

In [ ]:
# Add a group-level average back onto every row using transform
df["city_avg_salary"] = df.groupby("City")["Salary"].transform("mean").round(1)
df

## 11. Basic Feature Engineering

Create a few derived columns from the cleaned data.

In [ ]:
# Extract year/month from JoinDate
df["Join_Year"] = df["JoinDate"].dt.year
df["Join_Month"] = df["JoinDate"].dt.month

# Age group bins
df["Age_Group"] = pd.cut(df["Age"], bins=[0,25,35,60],
                          labels=["Young","Mid-career","Senior"])

# Salary relative to their city's average
df["Salary_vs_City_Avg"] = (df["Salary"] - df["city_avg_salary"]).round(1)

df

## 12. Final Check & Save Cleaned Data

In [ ]:
print("Final shape:", df.shape)
print("\nRemaining missing values:\n", df.isnull().sum())
df.info()

In [ ]:
df.to_csv("cleaned_data.csv", index=False)
print("Saved cleaned_data.csv")